# Self-Query Retrieval — Let the LLM Write Your Metadata Filters
### Advanced RAG Series (2026 Edition) · by Mayank Chugh

**What this notebook does:** builds a **Self-Query Retriever** over an air cargo shipment log. An LLM reads a plain-English question — *"temperature-controlled shipments into Hong Kong over 1,000 kg"* — and writes a structured query: a semantic search string **plus** a metadata filter on origin, destination, commodity, weight, delay, temperature control and priority. A basic similarity-search RAG is built first as the baseline.

**Dataset:** 12 sample air cargo shipments through Hong Kong (HKG). Shipment IDs, weights and delays are illustrative sample data written for this episode — not real flight or customer records.

| Stack | Choice |
|---|---|
| Framework | LangChain 1.x (`langchain_classic`, `langchain_core`, `langchain_community`) |
| Vector store | Chroma (local, in-memory) |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` (local, free) |
| LLM | Local `Qwen2.5-3B-Instruct` on T4 → Anthropic → OpenAI fallback, temperature 0 |
| Secrets | Colab Secrets via `userdata.get()` |

> ⚙️ **Runtime:** T4 GPU recommended. On CPU, add an Anthropic or OpenAI key to Colab Secrets.

## Part A — Setup

In [1]:
!pip install -q langchain langchain-classic langchain-community langchain-core langchain-chroma \
             langchain-huggingface langchain-anthropic langchain-openai \
             sentence-transformers lark "transformers>=4.51.0" accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
bigframes 2.51.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
holidays 0.108 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 4.0.0 which is incompatible.
ibis-framework 9.5.0 requires python-dateutil<3,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
google-cloud-bigquery 3.47.0 requires python-dateutil<3.0.0,>=2.8.2, but you have python-dateutil 4.0.0 which is incompatible.
betterproto 2.0.0b7 requires python-dateutil<3.0,>=2.8, but you have python-dateutil 4.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 require

### Credentials (Colab Secrets)

In [2]:
import os

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

for key in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY"]:
    value = secret(key)
    if value:
        os.environ[key] = value
    print(f"{key}: {'found' if value else 'not set'}")

ANTHROPIC_API_KEY: found
OPENAI_API_KEY: found


### Imports

In [3]:
import textwrap
import torch
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains.query_constructor.base import (
    AttributeInfo,
    StructuredQueryOutputParser,
    get_query_constructor_prompt,
)
from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
from langchain_community.query_constructors.chroma import ChromaTranslator

/tmp/ipykernel_433/3339309403.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.query_constructors.chroma import ChromaTranslator


### Embeddings — local, free

In [4]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print(len(embeddings.embed_query("hello")))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


### LLM — local HuggingFace → Anthropic → OpenAI

In [5]:
PROVIDER = "auto"
LOCAL_MODEL = "Qwen/Qwen2.5-3B-Instruct"
ANTHROPIC_MODEL = "claude-haiku-5-5"
OPENAI_MODEL = "gpt-4o-mini"

def load_llm(provider=PROVIDER):
    if provider in ("auto", "local") and torch.cuda.is_available():
        from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
        pipe = HuggingFacePipeline.from_model_id(
            model_id=LOCAL_MODEL,
            task="text-generation",
            device=0,
            pipeline_kwargs={"max_new_tokens": 512, "do_sample": False, "return_full_text": False},
        )
        return ChatHuggingFace(llm=pipe), f"local:{LOCAL_MODEL}"
    if provider in ("auto", "anthropic") and os.environ.get("ANTHROPIC_API_KEY"):
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model=ANTHROPIC_MODEL, temperature=0), f"anthropic:{ANTHROPIC_MODEL}"
    if provider in ("auto", "openai") and os.environ.get("OPENAI_API_KEY"):
        from langchain_openai import ChatOpenAI
        return ChatOpenAI(model=OPENAI_MODEL, temperature=0), f"openai:{OPENAI_MODEL}"
    raise RuntimeError("No LLM available: use a GPU runtime or add an API key to Colab Secrets.")

llm, llm_name = load_llm()
print("Using", llm_name)

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Using local:Qwen/Qwen2.5-3B-Instruct


### The data — 12 air cargo shipments (sample data)

In [6]:
from langchain_core.documents import Document

docs = [
    Document(page_content="Vaccine consignment kept at 2 to 8 degrees in an active cold-chain container with a temperature logger attached.",
             metadata={"shipment_id": "CX-1001", "origin": "HKG", "destination": "FRA", "commodity": "pharma", "weight_kg": 820, "delay_hours": 0.0, "temperature_controlled": True, "priority": "express"}),
    Document(page_content="Live lobsters packed in chilled boxes with damp seaweed, flown overnight for restaurant supply.",
             metadata={"shipment_id": "CX-1002", "origin": "YVR", "destination": "HKG", "commodity": "perishables", "weight_kg": 1450, "delay_hours": 3.5, "temperature_controlled": True, "priority": "express"}),
    Document(page_content="Lithium-ion battery packs for electric scooters, declared as class 9 dangerous goods.",
             metadata={"shipment_id": "CX-1003", "origin": "HKG", "destination": "LAX", "commodity": "dangerous_goods", "weight_kg": 2300, "delay_hours": 6.0, "temperature_controlled": False, "priority": "standard"}),
    Document(page_content="Cross-border e-commerce parcels: phone cases, charging cables and small electronics.",
             metadata={"shipment_id": "CX-1004", "origin": "HKG", "destination": "LHR", "commodity": "e-commerce", "weight_kg": 3100, "delay_hours": 1.0, "temperature_controlled": False, "priority": "standard"}),
    Document(page_content="A racehorse travelling in a dedicated stall with its groom on board.",
             metadata={"shipment_id": "CX-1005", "origin": "DXB", "destination": "HKG", "commodity": "live_animals", "weight_kg": 600, "delay_hours": 0.5, "temperature_controlled": False, "priority": "express"}),
    Document(page_content="Fresh cherries pre-cooled before loading and kept chilled for the holiday fruit market.",
             metadata={"shipment_id": "CX-1006", "origin": "SCL", "destination": "HKG", "commodity": "perishables", "weight_kg": 5200, "delay_hours": 8.0, "temperature_controlled": True, "priority": "express"}),
    Document(page_content="Insulin pens packed in passive thermal boxes for a hospital pharmacy.",
             metadata={"shipment_id": "CX-1007", "origin": "HKG", "destination": "SIN", "commodity": "pharma", "weight_kg": 240, "temperature_controlled": True, "priority": "express"}),
    Document(page_content="Gold bars and banknotes moved under armed escort and held in the valuables strong room.",
             metadata={"shipment_id": "CX-1008", "origin": "ZRH", "destination": "HKG", "commodity": "valuables", "weight_kg": 150, "delay_hours": 0.0, "temperature_controlled": False, "priority": "express"}),
    Document(page_content="Spare jet engine for an aircraft-on-ground repair, needed urgently to return a grounded plane to service.",
             metadata={"shipment_id": "CX-1009", "origin": "HKG", "destination": "NRT", "commodity": "general", "weight_kg": 4800, "delay_hours": 2.0, "temperature_controlled": False, "priority": "express"}),
    Document(page_content="Clinical trial blood samples packed in dry ice, declared under UN1845.",
             metadata={"shipment_id": "CX-1010", "origin": "HKG", "destination": "BOS", "commodity": "pharma, dangerous_goods", "weight_kg": 95, "delay_hours": 4.0, "temperature_controlled": True, "priority": "express"}),
    Document(page_content="Seasonal fashion garments shipped on hanging rails for a store launch.",
             metadata={"shipment_id": "CX-1011", "origin": "HKG", "destination": "CDG", "commodity": "general", "weight_kg": 1900, "delay_hours": 0.0, "temperature_controlled": False, "priority": "standard"}),
    Document(page_content="Tropical aquarium fish in oxygenated bags inside insulated boxes.",
             metadata={"shipment_id": "CX-1012", "origin": "BKK", "destination": "HKG", "commodity": "live_animals", "weight_kg": 310, "delay_hours": 1.5, "temperature_controlled": True, "priority": "standard"}),
]
len(docs)

12

## Part B — Basic RAG (the baseline)

In [7]:
basic_store = Chroma.from_documents(docs, embeddings, collection_name="cargo_basic")
basic_store._collection.count()

12

In [8]:
question1 = "Which temperature-controlled shipments came into Hong Kong weighing over 1,000 kg?"
question2 = "Which shipments were delayed by more than 1 hour?"
question3 = "Tell me about the shipment carrying a racehorse."

In [9]:
for d in basic_store.similarity_search(question1):
    print(d.metadata["shipment_id"], d.metadata["destination"], d.metadata["weight_kg"], "|", d.page_content[:55])

CX-1001 FRA 820 | Vaccine consignment kept at 2 to 8 degrees in an active
CX-1006 HKG 5200 | Fresh cherries pre-cooled before loading and kept chill
CX-1004 LHR 3100 | Cross-border e-commerce parcels: phone cases, charging 
CX-1011 CDG 1900 | Seasonal fashion garments shipped on hanging rails for 


In [10]:
for d in basic_store.similarity_search(question3):
    print(d.metadata["shipment_id"], "|", d.page_content[:55])

CX-1005 | A racehorse travelling in a dedicated stall with its gr
CX-1008 | Gold bars and banknotes moved under armed escort and he
CX-1011 | Seasonal fashion garments shipped on hanging rails for 
CX-1009 | Spare jet engine for an aircraft-on-ground repair, need


In [11]:
rag_prompt = ChatPromptTemplate.from_template(
    """You are an air cargo operations assistant. Answer using only the shipments below.

{context}

Question: {question}
"""
)

def format_docs(documents):
    return "\n\n".join(f"{d.page_content} | metadata: {d.metadata}" for d in documents)

def wrap_text(text, width=90):
    return "\n".join(textwrap.fill(line, width=width) for line in text.split("\n"))

basic_retriever = basic_store.as_retriever(search_kwargs={"k": 3})

basic_chain = (
    {"context": basic_retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

In [12]:
print(wrap_text(basic_chain.invoke(question2)))

The shipment with shipment ID CX-1002 was delayed by 3.5 hours, which is more than 1 hour.
Therefore, the shipment with shipment ID CX-1002 was delayed by more than 1 hour.


## Part C — Self-Query Retrieval

In [13]:
metadata_field_info = [
    AttributeInfo(name="origin", description="IATA code of the departure airport, e.g. 'HKG'", type="string"),
    AttributeInfo(name="destination", description="IATA code of the arrival airport, e.g. 'HKG'", type="string"),
    AttributeInfo(name="commodity",
                  description="One of ['pharma', 'perishables', 'dangerous_goods', 'live_animals', 'e-commerce', 'valuables', 'general']",
                  type="string"),
    AttributeInfo(name="weight_kg", description="Chargeable weight in kilograms", type="integer"),
    AttributeInfo(name="delay_hours", description="Hours the shipment arrived late; 0 means on time", type="float"),
    AttributeInfo(name="temperature_controlled", description="True if the shipment needs a temperature-controlled chain", type="boolean"),
    AttributeInfo(name="priority", description="One of ['express', 'standard']", type="string"),
]
document_content_description = "Description of an air cargo shipment"

In [14]:
sq_prompt = get_query_constructor_prompt(document_content_description, metadata_field_info)
print(sq_prompt.input_variables)
print(sq_prompt.format(query="dummy question"))

['query']
Your goal is to structure the user's query to match the request schema provided below.

<< Structured Request Schema >>
When responding use a markdown code snippet with a JSON object formatted in the following schema:

```json
{
    "query": string \ text string to compare to document contents
    "filter": string \ logical condition statement for filtering documents
}
```

The query string should contain only text that is expected to match the contents of documents. Any conditions in the filter should not be mentioned in the query as well.

A logical condition statement is composed of one or more comparison and logical operation statements.

A comparison statement takes the form: `comp(attr, val)`:
- `comp` (eq | ne | gt | gte | lt | lte | contain | like | in | nin): comparator
- `attr` (string):  name of attribute to apply the comparison to
- `val` (string): is the comparison value

A logical operation statement takes the form `op(statement1, statement2, ...)`:
- `op` (and 

In [15]:
output_parser = StructuredQueryOutputParser.from_components()
query_constructor = sq_prompt | llm | output_parser

In [16]:
structured = query_constructor.invoke({"query": question1})
structured

StructuredQuery(query='temperature controlled and origin == HKG and weight_kg > 1000', filter=None, limit=None)

In [17]:
ChromaTranslator().visit_structured_query(structured)

('temperature controlled and origin == HKG and weight_kg > 1000', {})

In [18]:
sq_store = Chroma.from_documents(docs, embeddings, collection_name="cargo_selfquery")

retriever = SelfQueryRetriever(
    query_constructor=query_constructor,
    vectorstore=sq_store,
    structured_query_translator=ChromaTranslator(),
    search_kwargs={"k": 20},
)

In [19]:
for d in retriever.invoke(question1):
    m = d.metadata
    print(m["shipment_id"], m["origin"], "→", m["destination"], m["weight_kg"], "kg |", d.page_content[:45])

CX-1001 HKG → FRA 820 kg | Vaccine consignment kept at 2 to 8 degrees in
CX-1006 SCL → HKG 5200 kg | Fresh cherries pre-cooled before loading and 
CX-1010 HKG → BOS 95 kg | Clinical trial blood samples packed in dry ic
CX-1007 HKG → SIN 240 kg | Insulin pens packed in passive thermal boxes 
CX-1011 HKG → CDG 1900 kg | Seasonal fashion garments shipped on hanging 
CX-1012 BKK → HKG 310 kg | Tropical aquarium fish in oxygenated bags ins
CX-1008 ZRH → HKG 150 kg | Gold bars and banknotes moved under armed esc
CX-1002 YVR → HKG 1450 kg | Live lobsters packed in chilled boxes with da
CX-1003 HKG → LAX 2300 kg | Lithium-ion battery packs for electric scoote
CX-1004 HKG → LHR 3100 kg | Cross-border e-commerce parcels: phone cases,
CX-1005 DXB → HKG 600 kg | A racehorse travelling in a dedicated stall w
CX-1009 HKG → NRT 4800 kg | Spare jet engine for an aircraft-on-ground re


In [20]:
for d in retriever.invoke(question2):
    print(d.metadata["shipment_id"], d.metadata.get("delay_hours"), "h |", d.page_content[:50])

CX-1003 6.0 h | Lithium-ion battery packs for electric scooters, d
CX-1009 2.0 h | Spare jet engine for an aircraft-on-ground repair,
CX-1012 1.5 h | Tropical aquarium fish in oxygenated bags inside i
CX-1002 3.5 h | Live lobsters packed in chilled boxes with damp se
CX-1006 8.0 h | Fresh cherries pre-cooled before loading and kept 
CX-1010 4.0 h | Clinical trial blood samples packed in dry ice, de


In [21]:
self_query_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)
print("BASIC:\n", wrap_text(basic_chain.invoke(question2)))
print("\nSELF-QUERY:\n", wrap_text(self_query_chain.invoke(question2)))

BASIC:
 The shipment that was delayed by more than 1 hour is:

- **CX-1002**: Live lobsters packed in chilled boxes with damp seaweed, flown overnight
for restaurant supply. It was delayed by 3.5 hours.

SELF-QUERY:
 The shipments that were delayed by more than 1 hour are:

1. CX-1003 - Lithium-ion battery packs for electric scooters, delayed by 6.0 hours.
2. CX-1009 - Spare jet engine for an aircraft-on-ground repair, delayed by 2.0 hours
(though it's marked as "express" priority, the delay is still greater than 1 hour).
3. CX-1010 - Clinical trial blood samples packed in dry ice, delayed by 4.0 hours.

Therefore, CX-1003 and CX-1010 were delayed by more than 1 hour.


## Part D — Three silent failures

In [22]:
for d in retriever.invoke("Which live animal shipments came from Sydney?"):
    print(d.metadata["shipment_id"], "|", d.page_content[:50])
print("done")

done


In [23]:
for d in retriever.invoke("Which shipments arrived on time, less than 1 hour late?"):
    print(d.metadata["shipment_id"], d.metadata.get("delay_hours"), "|", d.page_content[:50])

CX-1011 0.0 | Seasonal fashion garments shipped on hanging rails


In [24]:
for d in retriever.invoke("Show me all the pharma shipments."):
    print(d.metadata["shipment_id"], d.metadata["commodity"], "|", d.page_content[:50])

CX-1007 pharma | Insulin pens packed in passive thermal boxes for a
CX-1001 pharma | Vaccine consignment kept at 2 to 8 degrees in an a


### The one-line version

In [25]:
quick_retriever = SelfQueryRetriever.from_llm(
    llm,
    sq_store,
    document_content_description,
    metadata_field_info,
    structured_query_translator=ChromaTranslator(),
    search_kwargs={"k": 20},
)
for d in quick_retriever.invoke("Heavy shipments over 3 tonnes"):
    print(d.metadata["shipment_id"], d.metadata["weight_kg"], "|", d.page_content[:50])

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


CX-1006 5200 | Fresh cherries pre-cooled before loading and kept 
CX-1004 3100 | Cross-border e-commerce parcels: phone cases, char
CX-1009 4800 | Spare jet engine for an aircraft-on-ground repair,


## Closing